# Household Electric Power Consumption — 02_Feature_Engineering

## Overview
## **This notebook focuses on creating new features from the household electric power consumption dataset to improve data analysis and identify meaningful patterns in electricity usage.**

## **The feature engineering process includes extracting date and time features, categorizing day periods and seasons, identifying peak hours, calculating total and other energy consumption, deriving electrical features such as apparent power, power factor, impedance, and active-reactive power difference, categorizing consumption levels, and calculating sub-metering ratios and statistics.**


## Workflow

00. **Import Data**
01. **Date and Time Feature Engineering**
02. **Day Period**
03. **Season**
04. **IsPeakHour**
05. **Sub_metering_total**
06. **Other_energy_consumption**
07. **Power Conversion and Remaining Energy**
08. **Apparent Power**
09. **Impedance**
10. **Active_Reactive_Difference**
11. **Consumption_Level**
12. **Sub-metering Ratios**
13. **Min and Max Sub-metering**
14. **Reactive Power Ratio**

In [1]:
import pandas as pd
import numpy as np

# 00. Import Data

In [2]:
data = pd.read_csv('data_cleaned.csv')

In [3]:
data

,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,2010-11-26 21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2049280 entries, 0 to 2049279
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Datetime               str    
 1   Global_active_power    float64
 2   Global_reactive_power  float64
 3   Voltage                float64
 4   Global_intensity       float64
 5   Sub_metering_1         float64
 6   Sub_metering_2         float64
 7   Sub_metering_3         float64
dtypes: float64(7), str(1)
memory usage: 162.2 MB


# 01. Date and Time Feature Engineering

Extracting date and time features from the `Datetime` column to analyze energy consumption patterns.

- `Year`: Year of the reading.
- `Quarter`: Quarter of the year (1–4).
- `Month`: Month number (1–12).
- `Day`: Day of the month (1–31).
- `Hour`: Hour of the day (0–23).
- `WeekOfYear`: Week number of the year.
- `DayOfWeek`: Day number (Monday = 0, Sunday = 6).
- `DayName`: Name of the day.
- `MonthName`: Name of the month.
- `IsWeekend`: Whether the day is Saturday or Sunday.
- `IsMonthStart`: 1 if the date is the first day of the month, otherwise 0.
- `IsMonthEnd`: 1 if the date is the last day of the month, otherwise 0.

In [5]:
data["Datetime"] = pd.to_datetime(data["Datetime"], errors="coerce")

In [6]:
data["Year"] = data["Datetime"].dt.year
data["Quarter"] = data["Datetime"].dt.quarter
data["Month"] = data["Datetime"].dt.month
data['Day'] = data["Datetime"].dt.day
data["Hour"] = data["Datetime"].dt.hour
data["WeekOfYear"] = data["Datetime"].dt.isocalendar().week
data["DayOfWeek"] = data["Datetime"].dt.dayofweek
data["DayName"] = data["Datetime"].dt.day_name()
data["MonthName"] = data["Datetime"].dt.month_name()
data["IsWeekend"] = data["DayOfWeek"].isin([5, 6])

data['IsMonthStart'] = data['Datetime'].dt.is_month_start.astype(int) #is the month start or not, 1 for month start, 0 for not
data['IsMonthEnd'] = data['Datetime'].dt.is_month_end.astype(int)

# 02. Day Period
Categorizes each reading into Night, Morning, Afternoon, or Evening based on the hour.

In [7]:
data["Day_Period"] = pd.cut(data["Datetime"].dt.hour , bins=[-1, 5, 11, 17, 23] , labels=["Night", "Morning", "Afternoon", "Evening"])

# 03. Season
Categorizes each reading into Winter, Spring, Summer, or Autumn based on the month.

In [8]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

data['Season'] = data['Month'].apply(get_season)

# 04. IsPeakHour
Identifies peak hours between 5 PM and 10 PM (1 = peak, 0 = non-peak).

In [9]:
data['IsPeakHour'] = data['Hour'].between(17, 22).astype(int)

# 05. Sub_metering_total
Calculates the total energy consumption from the three sub-metering measurements.

In [10]:
data["Sub_metering_total"] = (data["Sub_metering_1"] + data["Sub_metering_2"] + data["Sub_metering_3"])

# 06. Other_energy_consumption
Calculates energy consumption not covered by the three sub-metering measurements, in Wh per minute.

In [11]:
data['Other_energy_consumption'] = (data['Global_active_power'] * 1000 / 60 - data['Sub_metering_total'])

# 07. Power Conversion and Remaining Energy
Converts active and reactive power into Wh per minute and calculates the remaining energy after subtracting total sub-metering consumption.

In [12]:
data['Global_Reactive_Power_Watt'] = data['Global_reactive_power'] * 1000 / 60
data['Global_Active_Power_Watt'] = data['Global_active_power'] * 1000 / 60
data['Remaining_Energy'] = data['Global_Active_Power_Watt'] - data['Sub_metering_total']

# 08. Apparent Power
Calculates apparent power using voltage and current.

### Power Factor
Measures the ratio of active power to apparent power. A value closer to 1 generally indicates more effective power usage.

In [13]:
data['Apparent_Power'] = data['Voltage'] * data['Global_intensity']

data['Power_Factor'] = np.where(
    data['Apparent_Power'] > 0,
    data['Global_Active_Power_Watt'] / data['Apparent_Power'],
    0)      #the more the power factor is close to 1, the more efficient the electric devices is.

# 09. Impedance
Calculates the ratio of voltage to current to estimate electrical impedance.

In [14]:
data['Impedance'] = np.where(data['Global_intensity'] > 0 , data['Voltage'] / data['Global_intensity'], 0)

# 10. Active_Reactive_Difference
Calculates the difference between active and reactive power.

In [15]:
data["Active_Reactive_Difference"] = (data["Global_active_power"] - data["Global_reactive_power"])

# 11. Consumption_Level
Categorizes energy consumption into Low, Medium, or High based on active power.

In [16]:
bins = [-float('inf'), 0.5, 2.0, float('inf')]
labels = ['Low', 'Medium', 'High']
data['Consumption_Level'] = pd.cut(data['Global_active_power'], bins=bins, labels=labels)

# 12. Sub-metering Ratios
Calculate the proportion of total sub-metering consumption for each of the three sub-meters.

In [17]:
data['Sub_1_Ratio'] = data['Sub_metering_1'] / (data['Sub_metering_total'] + 0.001)
data['Sub_2_Ratio'] = data['Sub_metering_2'] / (data['Sub_metering_total'] + 0.001)
data['Sub_3_Ratio'] = data['Sub_metering_3'] / (data['Sub_metering_total'] + 0.001)

# 13. Min and Max Sub-metering
Identify the minimum and maximum consumption values among the three sub-meters for each reading.

In [18]:
data["Min_Sub_metering"] = data[["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]].min(axis=1)
data["Max_Sub_metering"] = data[["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]].max(axis=1)

# 14. Reactive Power Ratio
Calculates the ratio of reactive power to active power.

In [19]:
data["Reactive_Power_Ratio"] = (data["Global_reactive_power"] / data["Global_active_power"].replace(0, float("nan")))

In [20]:
data.head(10)

,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Year,Quarter,...,Power_Factor,Impedance,Active_Reactive_Difference,Consumption_Level,Sub_1_Ratio,Sub_2_Ratio,Sub_3_Ratio,Min_Sub_metering,Max_Sub_metering,Reactive_Power_Ratio
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006,4,...,0.016261,12.763043,3.798,High,0.0,0.055552,0.944392,0.0,17.0,0.099146
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006,4,...,0.016625,10.157826,4.924,High,0.0,0.058820,0.941121,0.0,16.0,0.081343
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006,4,...,0.016693,10.143043,4.876,High,0.0,0.105258,0.894690,0.0,17.0,0.092668
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006,4,...,0.016704,10.162609,4.886,High,0.0,0.055552,0.944392,0.0,17.0,0.093170
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006,4,...,0.016408,14.916456,3.138,High,0.0,0.055552,0.944392,0.0,17.0,0.144026
5,2006-12-16 17:29:00,3.520,0.522,235.02,15.0,0.0,2.0,17.0,2006,4,...,0.016642,15.668000,2.998,High,0.0,0.105258,0.894690,0.0,17.0,0.148295
6,2006-12-16 17:30:00,3.702,0.520,235.09,15.8,0.0,1.0,17.0,2006,4,...,0.016611,14.879114,3.182,High,0.0,0.055552,0.944392,0.0,17.0,0.140465
7,2006-12-16 17:31:00,3.700,0.520,235.22,15.8,0.0,1.0,17.0,2006,4,...,0.016593,14.887342,3.180,High,0.0,0.055552,0.944392,0.0,17.0,0.140541
8,2006-12-16 17:32:00,3.668,0.510,233.99,15.8,0.0,1.0,17.0,2006,4,...,0.016536,14.809494,3.158,High,0.0,0.055552,0.944392,0.0,17.0,0.139040
9,2006-12-16 17:33:00,3.662,0.510,233.86,15.8,0.0,2.0,16.0,2006,4,...,0.016518,14.801266,3.152,High,0.0,0.111105,0.888840,0.0,16.0,0.139268


In [21]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2049280 entries, 0 to 2049279
Data columns (total 39 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   Datetime                    datetime64[us]
 1   Global_active_power         float64       
 2   Global_reactive_power       float64       
 3   Voltage                     float64       
 4   Global_intensity            float64       
 5   Sub_metering_1              float64       
 6   Sub_metering_2              float64       
 7   Sub_metering_3              float64       
 8   Year                        int32         
 9   Quarter                     int32         
 10  Month                       int32         
 11  Day                         int32         
 12  Hour                        int32         
 13  WeekOfYear                  UInt32        
 14  DayOfWeek                   int32         
 15  DayName                     str           
 16  MonthName                   s